In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/keypoint_tennis_court/tennis_court_det_dataset.zip

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  inflating: data/images/JNKp7sCdQlY_2200.png  
  inflating: data/images/JNKp7sCdQlY_2250.png  
  inflating: data/images/JNKp7sCdQlY_250.png  
  inflating: data/images/JNKp7sCdQlY_300.png  
  inflating: data/images/JNKp7sCdQlY_350.png  
  inflating: data/images/JNKp7sCdQlY_400.png  
  inflating: data/images/JNKp7sCdQlY_450.png  
  inflating: data/images/JNKp7sCdQlY_50.png  
  inflating: data/images/JNKp7sCdQlY_500.png  
  inflating: data/images/JNKp7sCdQlY_550.png  
  inflating: data/images/JNKp7sCdQlY_600.png  
  inflating: data/images/JNKp7sCdQlY_650.png  
  inflating: data/images/JNKp7sCdQlY_700.png  
  inflating: data/images/JNKp7sCdQlY_750.png  
  inflating: data/images/juXbdW7z0WA_100.png  
  inflating: data/images/juXbdW7z0WA_1050.png  
  inflating: data/images/juXbdW7z0WA_1100.png  
  inflating: data/images/juXbdW7z0WA_200.png  
  inflating: data/images/juXbdW7z0WA_350.png  
  inflating: data/images/juXbdW7z0WA_400.png  


In [ ]:
import os

os.listdir('/content/data')

['images', 'data_train.json', 'data_val.json']

In [ ]:
import json
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection.keypoint_rcnn import KeypointRCNNPredictor
from torchvision.transforms import functional as F
from PIL import Image

In [ ]:
class TennisCourtDataset(Dataset):
    def __init__(self, root_dir, json_file, img_ext='.png', max_samples=None):
        self.root_dir = root_dir
        self.img_ext = img_ext

        with open(json_file, 'r') as f:
            self.data = json.load(f)

        # Limit the amount of data
        if max_samples is not None:
            self.data = self.data[:max_samples]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img_name = str(item['id']) + self.img_ext
        img_path = os.path.join(self.root_dir, 'images', img_name)

        image = Image.open(img_path).convert("RGB")
        image = F.to_tensor(image)

        kps_raw = item['kps']
        keypoints = []
        x_coords = []
        y_coords = []

        for kp in kps_raw:
            x, y = kp[0], kp[1]
            keypoints.append([x, y, 1])
            x_coords.append(x)
            y_coords.append(y)

        xmin = max(0, min(x_coords) - 5)
        xmax = max(x_coords) + 5
        ymin = max(0, min(y_coords) - 5)
        ymax = max(y_coords) + 5

        boxes = torch.as_tensor([[xmin, ymin, xmax, ymax]], dtype=torch.float32)
        labels = torch.ones((1,), dtype=torch.int64)
        keypoints = torch.as_tensor([keypoints], dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": labels,
            "keypoints": keypoints
        }

        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
def get_model(num_keypoints):
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.keypoint_predictor.kps_score_lowres.in_channels  # đổi kps_score_fcn -> kps_score_lowres
    model.roi_heads.keypoint_predictor = KeypointRCNNPredictor(in_features, num_keypoints)
    return model

In [ ]:
import json
import os
import torch
from torch.utils.data import DataLoader

# Khởi tạo đường dẫn
data_dir = '/content/data'
train_json = os.path.join(data_dir, 'data_train.json')
val_json = os.path.join(data_dir, 'data_val.json')

# Đọc toàn bộ dữ liệu từ file train và val gốc để chia lại thành 3 tập
with open(train_json, 'r') as f:
    full_train_data = json.load(f)
with open(val_json, 'r') as f:
    full_val_data = json.load(f)

# Gộp chung lại để chia chuẩn tỉ lệ Train/Val/Test (ví dụ: 80% / 10% / 10%)
all_data = full_train_data + full_val_data
total_samples = len(all_data)

train_split = int(0.8 * total_samples)
val_split = int(0.9 * total_samples)

# Chia dữ liệu
train_data_list = all_data[:train_split]
val_data_list = all_data[train_split:val_split]
test_data_list = all_data[val_split:]

# Ghi tạm các tập dữ liệu ra file json mới
with open('/content/data/custom_train.json', 'w') as f:
    json.dump(train_data_list, f)
with open('/content/data/custom_val.json', 'w') as f:
    json.dump(val_data_list, f)
with open('/content/data/custom_test.json', 'w') as f:
    json.dump(test_data_list, f)

# Khởi tạo các dataset tương ứng
train_dataset = TennisCourtDataset(data_dir, '/content/data/custom_train.json', max_samples=10000)
val_dataset = TennisCourtDataset(data_dir, '/content/data/custom_val.json', max_samples=2000)
test_dataset = TennisCourtDataset(data_dir, '/content/data/custom_test.json', max_samples=2000)

# Khởi tạo DataLoader
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

num_keypoints = 14
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Đang sử dụng device: {device}")
print(f"Số lượng ảnh Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Load model
model = get_model(num_keypoints)
model.to(device)

# Khởi tạo Optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Cấu hình Hyperparameter & Early Stopping
num_epochs = 50
best_val_loss = float('inf')


Đang sử dụng device: cuda
Số lượng ảnh Train: 7072 | Val: 884 | Test: 885
Downloading: "https://download.pytorch.org/models/keypointrcnn_resnet50_fpn_coco-fc266e95.pth" to /root/.cache/torch/hub/checkpoints/keypointrcnn_resnet50_fpn_coco-fc266e95.pth


100%|██████████| 226M/226M [00:01<00:00, 134MB/s]


In [ ]:
import os
import torch
from tqdm.auto import tqdm


checkpoint_path = "/content/drive/MyDrive/keypoint_tennis_court/last_tennis_court_checkpoint.pth"
best_checkpoint_path = "/content/drive/MyDrive/keypoint_tennis_court/best_tennis_court_checkpoint.pth"

# Khởi tạo các giá trị mặc định khi train từ đầu
start_epoch = 0
best_val_loss = float("inf")
trigger_times = 0

# --- KIỂM TRA VÀ RESUME (NẾU CÓ) ---
if os.path.exists(checkpoint_path):
  print(f"Phát hiện checkpoint cũ tại '{checkpoint_path}', đang tiến hành load...")
  checkpoint = torch.load(checkpoint_path)

  model.load_state_dict(checkpoint["model_state_dict"])
  optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
  start_epoch = (
      checkpoint["epoch"] + 1
  )  # Bắt đầu từ epoch tiếp theo của lần train trước
  best_val_loss = checkpoint["best_val_loss"]
  trigger_times = checkpoint.get("trigger_times", 0)

  print(
      f"Resume thành công! Tiếp tục huấn luyện từ Epoch {start_epoch + 1}"
      f"/{num_epochs}"
  )
else:
  print("Không tìm thấy checkpoint cũ. Bắt đầu huấn luyện từ đầu...")

model.to(device)

print("Bắt đầu huấn luyện...")

for epoch in range(start_epoch, num_epochs):
  model.train()
  train_loss = 0

  train_bar = tqdm(
      train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]"
  )

  for images, targets in train_bar:
    images = [image.to(device) for image in images]
    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

    optimizer.zero_grad()
    loss_dict = model(images, targets)
    losses = sum(loss for loss in loss_dict.values())

    losses.backward()
    optimizer.step()

    train_loss += losses.item()
    train_bar.set_postfix(loss=losses.item())

  avg_train_loss = train_loss / len(train_loader)

  # --- Validation ---
  val_loss = 0
  model.train()  # Model cần ở chế độ train() để lấy được losses từ RCNN

  val_bar = tqdm(
      val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False
  )

  with torch.no_grad():
    for images, targets in val_bar:
      images = [image.to(device) for image in images]
      targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

      loss_dict = model(images, targets)
      losses = sum(loss for loss in loss_dict.values())
      val_loss += losses.item()
      val_bar.set_postfix(loss=losses.item())

  avg_val_loss = val_loss / len(val_loader)

  print(
      f"\nKết quả Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f} | Val Loss:"
      f" {avg_val_loss:.4f}"
  )

  # --- Lưu Model Tốt Nhất (Best Model) ---
  if avg_val_loss < best_val_loss:
    best_val_loss = avg_val_loss
    trigger_times = 0

    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "trigger_times": trigger_times,
        },
        best_checkpoint_path,
    )
    print(f"  -> Đã lưu checkpoint model tốt nhất (Best)!")
  else:
    break

  # --- Lưu Trạng Thái Mới Nhất Sau Mỗi Epoch (Last Checkpoint) ---
  torch.save(
      {
          "epoch": epoch,
          "model_state_dict": model.state_dict(),
          "optimizer_state_dict": optimizer.state_dict(),
          "best_val_loss": best_val_loss,
          "trigger_times": trigger_times,
      },
      checkpoint_path,
  )

print("Hoàn tất huấn luyện!")

Không tìm thấy checkpoint cũ. Bắt đầu huấn luyện từ đầu...
Bắt đầu huấn luyện...


Epoch 1/50 [Train]:   0%|          | 0/884 [00:00<?, ?it/s]

Epoch 1/50 [Val]:   0%|          | 0/111 [00:00<?, ?it/s]


Kết quả Epoch 1: Train Loss: 1.1031 | Val Loss: 0.7910
  -> Đã lưu checkpoint model tốt nhất (Best)!


Epoch 2/50 [Train]:   0%|          | 0/884 [00:00<?, ?it/s]

...

Epoch 5/50 [Train]:   0%|          | 0/884 [00:00<?, ?it/s]

Epoch 5/50 [Val]:   0%|          | 0/111 [00:00<?, ?it/s]


Kết quả Epoch 5: Train Loss: 0.6277 | Val Loss: 0.5673
  -> Đã lưu checkpoint model tốt nhất (Best)!


Epoch 6/50 [Train]:   0%|          | 0/884 [00:00<?, ?it/s]

Epoch 6/50 [Val]:   0%|          | 0/111 [00:00<?, ?it/s]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

model.load_state_dict(torch.load('best_tennis_court_keypoint_model.pth'))
model.eval()
@torch.no_grad()
def show_predictions(model, dataset, device, n_samples=6, score_threshold=0.5):
    model.eval()
    indices = np.random.choice(len(dataset), size=n_samples, replace=False)

    fig, axes = plt.subplots(1, n_samples, figsize=(n_samples * 4, 5))
    if n_samples == 1:
        axes = [axes]

    for ax, idx in zip(axes, indices):
        image_tensor, target = dataset[idx]
        gt_keypoints = target["keypoints"][0, :, :2].numpy()

        output = model([image_tensor.to(device)])[0]

        img_display = image_tensor.permute(1, 2, 0).cpu().numpy()
        ax.imshow(img_display)
        ax.scatter(gt_keypoints[:, 0], gt_keypoints[:, 1], c="lime", s=30, marker="o", label="GT")

        if len(output["scores"]) > 0:
            best_idx = output["scores"].argmax().item()
            if output["scores"][best_idx].item() >= score_threshold:
                pred_keypoints = output["keypoints"][best_idx][:, :2].cpu().numpy()
                ax.scatter(pred_keypoints[:, 0], pred_keypoints[:, 1], c="red", s=30, marker="x", label="Pred")

                box = output["boxes"][best_idx].cpu().numpy()
                rect = patches.Rectangle(
                    (box[0], box[1]), box[2] - box[0], box[3] - box[1],
                    linewidth=1.5, edgecolor="yellow", facecolor="none",
                )
                ax.add_patch(rect)

        ax.axis("off")

    axes[0].legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()


show_predictions(model, test_dataset, device, n_samples=6)
